In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")
home = Path("/content/drive/MyDrive/colab/thesis/agenticsumm")

In [ ]:
import re
import numpy as np
import pandas as pd

def format_text(text: str):
    return " ".join(
        re.sub(r"^\s*\d+\.\s*", "", line)
        for line in text.splitlines()
    )

test_hqs_df = pd.read_json(home / "hqs_test_agsumm.jsonl", lines=True)
test_rrs_df = pd.read_json(home / "rrs_test_agsumm.jsonl", lines=True)
test_rrs_df["agsumm"] = test_rrs_df["agsumm"].map(format_text)

# Basic Eval

In [ ]:
!pip install -q uv
!uv pip install evaluate rouge-score bert-score gdown
!uv pip install git+https://github.com/google-research/bleurt.git
!gdown "1pjd8TxqGeYVdCWZHdXow8pDXpwRebwFY"
!unzip ClinicalBLEURT.zip

import evaluate
from bert_score import BERTScorer
from bleurt.score import BleurtScorer

In [ ]:
# @title 🚀 Metrics
from types import SimpleNamespace

metric = SimpleNamespace()
metric.rouge = evaluate.load("rouge")
metric.bertscore = BERTScorer(model_type="allenai/scibert_scivocab_cased")
metric.bleurtscore = BleurtScorer(checkpoint="ClinicalBLEURT")


def compute_rouge(refs, preds):
    rouge = metric.rouge
    return rouge.compute(predictions=preds, references=refs)


def compute_bertscore(refs, preds):
    bertscore = metric.bertscore
    bertscore._tokenizer.model_max_length = 512
    P, R, F = bertscore.score(preds, refs)
    return {
        "bertscore_pr": P.mean().item(),
        "bertscore_re": R.mean().item(),
        "bertscore_f1": F.mean().item(),
    }


def compute_bleurtscore(refs, preds):
    bleurtscore = metric.bleurtscore
    result = bleurtscore.score(references=refs, candidates=preds)
    return { "clinicalbleurt": np.mean(result) }


def compute_metrics_all(refs, preds):
    return {
        **compute_rouge(refs, preds),
        **compute_bertscore(refs, preds),
        **compute_bleurtscore(refs, preds),
    }

In [ ]:
from tabulate import tabulate

headers = ["METRIC", "SCORE"]
results = compute_metrics_all(
    test_hqs_df["summary"].to_list(),
    test_hqs_df["agsumm"].to_list()
)
print("---HQS---")
print(tabulate(results.items(), headers, tablefmt="simple"))

results = compute_metrics_all(
    test_rrs_df["impression"].to_list(),
    test_rrs_df["agsumm"].to_list()
)
print("\n---RRS---")
print(tabulate(results.items(), headers, tablefmt="simple"))

# SummaC

In [ ]:
!pip install -q uv
!uv pip install summac sentencepiece

import nltk; nltk.download('punkt_tab')
from summac.model_summac import SummaCZS
model_zs = SummaCZS(granularity="document", model_name="vitc", device="cuda")

def compute_summac(sources, preds):
    scores_zs = model_zs.score(sources, preds)
    return np.mean(scores_zs["scores"])

results = compute_summac(
    test_hqs_df["source"].to_list(),
    test_hqs_df["agsumm"].to_list(),
)
print(f"SummaC Score for HQS: {results:.4f}")

results = compute_summac(
    test_rrs_df["findings"].to_list(),
    test_rrs_df["agsumm"].to_list(),
)
print(f"SummaC Score for RRS: {results:.4f}")

# Far and C_F1

In [ ]:
!pip install -q uv
!uv pip install scispacy
!uv pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_lg-0.5.4.tar.gz

---

[CTRL+M .]

---

In [ ]:
import spacy
import scispacy
import numpy as np
import pandas as pd
from pathlib import Path
from scispacy.abbreviation import AbbreviationDetector
from scispacy.linking import EntityLinker

In [ ]:
nlp = spacy.load("en_core_sci_lg", enable=["ner"])

# @title Computes Faithfulness-adjusted Recall (FaR) and Concept F1 (CF1)
def _extract_entities(text, nlp_model):
    doc = nlp_model(text)
    return set([
        word
        for ent in doc.ents
        for word in str(ent.text).strip().lower().split()
    ])


def _compute_metrics(source, reference, generated, nlp_model):
    # Step 1: Extract entities as sets for each document
    S = _extract_entities(source, nlp_model)
    R = _extract_entities(reference, nlp_model)
    M = _extract_entities(generated, nlp_model)

    # Step 2: Map to Venn diagram intersections
    # Region C: Entities shared by Source, Reference, and Generated
    C = S.intersection(R).intersection(M)

    # Regions B + C: Entities shared by Source and Reference
    B_plus_C = S.intersection(R)

    # Regions C + F: Entities shared by Reference and Generated
    C_plus_F = R.intersection(M)

    # Step 3: Compute Faithfulness-adjusted Recall (FaR)
    # FaR = C / (B + C)
    if len(B_plus_C) > 0:
        far_score = len(C) / len(B_plus_C)
    else:
        far_score = 0.0  # Handle division by zero

    # Step 4: Compute Concept Recall and Concept Precision
    # Concept Rec = (C + F) / (B + C + E + F)
    # Note: R is equivalent to regions B + C + E + F
    if len(R) > 0:
        concept_rec = len(C_plus_F) / len(R)
    else:
        concept_rec = 0.0

    # Concept Prec = (C + F) / (C + D + F + G)
    # Note: M is equivalent to regions C + D + F + G
    if len(M) > 0:
        concept_prec = len(C_plus_F) / len(M)
    else:
        concept_prec = 0.0

    # Step 5: Compute Concept F1
    if (concept_rec + concept_prec) > 0:
        concept_f1 = (2 * concept_rec * concept_prec) / (concept_rec + concept_prec)
    else:
        concept_f1 = 0.0

    return far_score, concept_f1

In [ ]:
def compute_metrics(sources, refs, preds, nlp_model):
    far_scores = []
    cf1_scores = []

    for source, ref, pred in zip(sources, refs, preds):
        far, cf1 = _compute_metrics(source, ref, pred, nlp_model)
        far_scores.append(far)
        cf1_scores.append(cf1)

    return np.mean(far_scores), np.mean(cf1_scores)

far, cf1 = compute_metrics(
    test_hqs_df["source"].to_list(),
    test_hqs_df["summary"].to_list(),
    test_hqs_df["agsumm"].to_list(),
    nlp_model=nlp
)

print(f"FaR Score of HQS: {far:.4f}")
print(f"Concept F1 Score of HQS: {cf1:.4f}")

far, cf1 = compute_metrics(
    test_rrs_df["findings"].to_list(),
    test_rrs_df["impression"].to_list(),
    test_rrs_df["agsumm"].to_list(),
    nlp_model=nlp
)

print(f"FaR Score of RRS: {far:.4f}")
print(f"Concept F1 Score of RRS: {cf1:.4f}")

# Misc

In [ ]:
!pip install uv
!uv pip install scispacy
!uv pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz

In [ ]:
# @title
import scispacy
import spacy

nlp = spacy.load("en_ner_bc5cdr_md")

def extract_medical_entities(text, model):
    """Extracts unique medical entities from text using a clinical NER model."""
    doc = model(text)

    # Normalize entities (lowercasing, stripping whitespace, removing duplicates)
    entities = set(ent.text.lower().strip() for ent in doc.ents)
    return entities

def evaluate_entity_preservation(source_text, summary_text):
    """Calculates Precision, Recall, and F1 for entity alignment between source and summary."""
    # Step 1: Extract entities
    source_entities = extract_medical_entities(source_text, nlp)
    summary_entities = extract_medical_entities(summary_text, nlp)

    # Step 2: Calculate intersections
    true_positives = source_entities.intersection(summary_entities)

    # Step 3: Handle edge cases where no entities are found
    if not source_entities:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0, "source_ents": [], "summary_ents": []}

    # Step 4: Calculate metrics
    precision = len(true_positives) / len(summary_entities) if summary_entities else 0.0
    recall = len(true_positives) / len(source_entities)

    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    return {
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "preserved_entities": list(true_positives),
        "hallurinated_entities": list(summary_entities - source_entities),
        "omitted_entities": list(source_entities - summary_entities)
    }

In [ ]:
from tqdm.auto import trange

results = []
for i in trange(len(test_hqs_df)):
    results.append(evaluate_entity_preservation(test_hqs_df["source"][i], test_hqs_df["agsumm"][i]))

In [ ]:
r = [i["recall"] for i in results if "source_ents" not in i]
sum(r) / len(r)

In [ ]:
from tqdm.auto import trange

results = []
for i in trange(len(test_rrs_df)):
    results.append(evaluate_entity_preservation(test_rrs_df["findings"][i], test_rrs_df["agsumm"][i]))

In [ ]:
r = [i["recall"] for i in results if "source_ents" not in i]
sum(r) / len(r)